# 01 - Single run vibration analysis

This notebook teaches the first full analysis loop for **one CSV recording**:
1. Load and validate data
2. Estimate sampling rate
3. Plot acceleration (X/Y/Z) in time domain
4. Compute basic metrics
5. Remove DC and run FFT
6. Inspect dominant frequency peaks


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from helpers import (
    add_magnitude,
    basic_metrics,
    compute_fft,
    dominant_peaks,
    estimate_sampling_rate_hz,
    load_measurement_csv,
)


## Choose input file
Edit this path to any measurement CSV in `data/raw/` or `data/sample/`.


In [ ]:
csv_path = Path('../data/sample/sample_run_01.csv')
df = load_measurement_csv(str(csv_path))
df = add_magnitude(df)
sample_rate_hz = estimate_sampling_rate_hz(df)
print(f'Samples: {len(df)}')
print(f'Estimated sample rate: {sample_rate_hz:.2f} Hz')
df.head()


## Time-domain plots
These show how acceleration varies over time for each axis.


In [ ]:
t = df['timestamp_ms'] / 1000.0
fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
ax[0].plot(t, df['ax_g'], label='ax_g')
ax[1].plot(t, df['ay_g'], label='ay_g', color='tab:orange')
ax[2].plot(t, df['az_g'], label='az_g', color='tab:green')
ax[3].plot(t, df['a_mag_g'], label='a_mag_g', color='tab:red')
for axis in ax:
    axis.legend(loc='upper right')
    axis.grid(True, alpha=0.3)
ax[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()


## Basic vibration metrics
Mean, RMS, peak, and standard deviation per acceleration axis.


In [ ]:
metrics = basic_metrics(df)
metrics


## Frequency-domain analysis (FFT)
We remove DC offset before FFT, then inspect dominant peaks.


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for i, col in enumerate(['ax_g', 'ay_g', 'az_g']):
    freqs, amps = compute_fft(df[col].to_numpy(), sample_rate_hz)
    ax[i].plot(freqs, amps, label=f'{col} spectrum')
    ax[i].set_ylabel('Amplitude')
    ax[i].grid(True, alpha=0.3)
    ax[i].legend(loc='upper right')
ax[-1].set_xlabel('Frequency [Hz]')
plt.tight_layout()
plt.show()

for col in ['ax_g', 'ay_g', 'az_g']:
    freqs, amps = compute_fft(df[col].to_numpy(), sample_rate_hz)
    peaks = dominant_peaks(freqs, amps, top_n=5)
    print(f'\nTop peaks for {col}:')
    print(peaks.to_string(index=False))


## Interpretation notes
- Check if one axis dominates (mount direction sensitivity).
- Compare peak frequencies with expected engine/excitation behavior.
- If low-frequency drift dominates, consider filtering in future stages.
